In [4]:
import torch
from torch.utils.data import DataLoader
from datasets import Dataset
from typing import Callable, List
from transformers import BertTokenizer, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification, PreTrainedTokenizer, PreTrainedModel

In [ ]:
def extract_features_from_layer(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    dataset: Dataset,
    tokenize_fn: Callable,
    target_layer: torch.nn.Module,
    batch_size: int = 16,
    max_length: int = 128,
    device: str = "cpu"
):
    
    model.to(device)
    model.eval()

    def tokenize_batch(batch):
        return tokenize_fn(tokenizer, batch, max_length)

    tokenized_data = dataset.map(tokenize_batch, batched=True)
    tokenized_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    dataloader = DataLoader(tokenized_data, batch_size=batch_size)

    extracted_features = []
    all_labels = []

    def hook_fn(module, input, output):
        extracted_features.append(output.detach().cpu())

    hook_handle = target_layer.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"]

            model(input_ids=input_ids, attention_mask=attention_mask)
            all_labels.append(labels)

    hook_handle.remove()

    features = torch.cat(extracted_features, dim=0)
    labels = torch.cat(all_labels, dim=0)

    return features, labels

In [6]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base")

print(bert_model)
print(roberta_model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [7]:
from datasets import load_dataset

rte_dataset = load_dataset("glue", "rte", split="train")
rte_dataset


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 2490
})

In [8]:
# Função de tokenização genérica para o dataset RTE
def tokenize_rte(tokenizer, batch, max_length=128):
    """Tokenização genérica para RTE (BERT, RoBERTa, etc)"""
    return tokenizer(
        batch["sentence1"], 
        batch["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

def tokenize_rte_longest(tokenizer, batch, max_length):
    return tokenizer(
        batch["sentence1"], 
        batch["sentence2"],
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8  
max_length = 128


bert_features, bert_labels = extract_features_from_layer(
    model=bert_model,
    tokenizer=bert_tokenizer,
    dataset=rte_dataset,
    tokenize_fn=tokenize_rte_longest,
    target_layer=bert_model.bert.pooler,  
    batch_size=batch_size,
    max_length=max_length,
    device=device
)

print(f"BERT Features shape: {bert_features.shape}") 
print(f"BERT Sample feature: {bert_features[0][:5]}") 


/home/dods/miniconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


BERT Features shape: torch.Size([2490, 768])
BERT Sample feature: tensor([-0.9175, -0.6920, -0.9580,  0.8733,  0.7515])


In [10]:
bert_features

tensor([[-0.9175, -0.6920, -0.9580,  ..., -0.8404, -0.7485,  0.8663],
        [-0.8652, -0.4237, -0.8453,  ..., -0.4934, -0.5822,  0.6933],
        [-0.8737, -0.5430, -0.9180,  ..., -0.7726, -0.6705,  0.6709],
        ...,
        [-0.8921, -0.4215, -0.5669,  ..., -0.6250, -0.5654,  0.6317],
        [-0.7553, -0.5337, -0.9195,  ..., -0.3084, -0.7167,  0.4566],
        [-0.8532, -0.6109, -0.9230,  ..., -0.7965, -0.6540,  0.6182]])

In [11]:
roberta_features, roberta_labels = extract_features_from_layer(
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    dataset=rte_dataset,
    tokenize_fn=tokenize_rte_longest,
    target_layer=roberta_model.classifier.dense,  
    batch_size=batch_size,
    max_length=max_length,
    device=device
)

print(f"RoBERTa Features shape: {roberta_features.shape}") 
print(f"RoBERTa Sample feature: {roberta_features[0]}")  

/home/dods/miniconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


RoBERTa Features shape: torch.Size([2490, 768])
RoBERTa Sample feature: tensor([ 1.9382e-01,  3.9789e-01,  1.6772e-01,  4.4294e-01, -1.2841e-01,
        -6.8609e-02,  4.1543e-01, -4.8756e-02, -4.4539e-02,  2.0045e-01,
         5.2180e-01, -1.6396e-01, -4.8382e-01,  9.6780e-03,  9.3157e-02,
         1.2493e-01, -1.3610e-01,  2.3392e-01,  1.1679e-01,  2.3457e-02,
        -3.4931e-01,  1.8217e-01, -2.3170e-01,  5.1999e-01, -1.2383e-01,
        -2.2158e-01,  3.1804e-02,  1.8296e-01, -3.4281e-01,  4.1668e-01,
        -6.1185e-02, -2.2683e-01,  7.0725e-02, -2.2649e-01,  3.9220e-02,
        -2.8408e-01,  3.0951e-02,  6.5879e-02, -2.9634e-01,  4.2652e-01,
        -4.7613e-01, -1.5959e-01,  2.4696e-01,  4.7403e-02, -3.5202e-01,
         4.0767e-02,  3.2713e-01,  1.7790e-01,  6.3065e-02, -2.1562e-02,
        -5.3246e-02,  9.0285e-02,  2.8134e-01, -5.1505e-01, -4.0975e-02,
        -3.7331e-02, -7.7536e-02,  5.4475e-01,  1.0241e-01, -1.3905e-01,
         2.1904e-01,  7.0159e-02,  2.7404e-01,  1.16

In [ ]:
roberta_features

tensor([[ 0.1938,  0.3979,  0.1677,  ..., -0.1126,  0.3067, -0.2085],
        [ 0.2055,  0.3813,  0.1526,  ..., -0.1363,  0.3059, -0.1947],
        [ 0.1929,  0.3886,  0.1688,  ..., -0.0863,  0.2553, -0.1866],
        ...,
        [ 0.2271,  0.4084,  0.2028,  ..., -0.0997,  0.3007, -0.1818],
        [ 0.2132,  0.4221,  0.1952,  ..., -0.1233,  0.2783, -0.1681],
        [ 0.1876,  0.3983,  0.1825,  ..., -0.1190,  0.3031, -0.1967]])

In [13]:
roberta_features.shape

torch.Size([2490, 768])

In [14]:
save_features = True  # Mude para True se quiser salvar

if save_features:
    import os
    
    # Criar diretório se não existir
    os.makedirs("extracted_features", exist_ok=True)
    
    # Salvar features
    torch.save({
        'bert_features': bert_features,
        'bert_labels': bert_labels,
        'roberta_features': roberta_features,
        'roberta_labels': roberta_labels,
        'dataset_info': {
            'name': 'RTE',
            'size': len(rte_dataset),
            'bert_layer': 'bert.pooler',
            'roberta_layer': 'classifier.dense'
        }
    }, 'extracted_features/rte_features.pt')

## Extração Meta-features

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pymfe.mfe import MFE
import torch

def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def ensure_dir(path: str | Path) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

In [23]:
try:
    valid_feats = set(MFE.valid_metafeatures(groups=["statistical"]))
    if not valid_feats:
        valid_feats = set(MFE.valid_metafeatures())
except TypeError:
    valid_feats = set(MFE.valid_metafeatures())

In [ ]:
def extract_metafeatures_with_pymfe(
    X,
    y,
    *,
    dataset_name: str = "RTE",
    split: str = "train",
    repr_name: str = "unknown",
    out_dir: str = "meta",
    fast_mode: bool = True,           # True = conjunto enxuto (rápido)
    include_landmarkers: bool = False,
    random_state: int = 42,
    verbose: bool = True,
):
    import numpy as np, pandas as pd
    from pathlib import Path
    from sklearn.preprocessing import StandardScaler
    from pymfe.mfe import MFE

    # --- helpers esperados já definidos no seu notebook ---
    X = to_numpy(X)
    y = to_numpy(y).astype(int).ravel()
    n, d = X.shape

    # 1) padronização
    X_std = StandardScaler().fit_transform(X)

    # 2) grupos
    groups = ["general", "statistical", "complexity"]
    if include_landmarkers:
        groups += ["landmarking"]

    # 3) descobrir o que a SUA versão do PyMFE suporta
    try:
        valid_all = set(MFE.valid_metafeatures(groups=groups))
    except TypeError:
        # versões antigas não aceitam 'groups=' aqui
        valid_all = set(MFE.valid_metafeatures())

    # conjunto prioritário (vamos filtrar pelo que existe na sua versão)
    priority_complex = {
        "f1","f2","f3","f4","n1","n2","n3","n4","l3","t2","t3","t4"
    }
    priority_general_stats = {
        # gerais
        "nr_inst","nr_attr","attr_to_inst","nr_class","class_ent",
        # estatística
        "mean","sd","var","skewness","kurtosis","cor","cov",
        # alguns nomes comuns de PCA (nem todas versões têm todos)
        "pca_first_eigv_var_ratio","pca_fraction","pca_frac1",
        "pca_worst","pca_best"
    }

    if fast_mode:
        # pega só o que estiver disponível
        feats = list((priority_complex | priority_general_stats) & valid_all)
        # fallback: se por algum motivo ficou vazio, usa tudo que está disponível
        if not feats:
            feats = list(valid_all)
    else:
        # modo completo: usa todas as features válidas dos grupos
        feats = list(valid_all)

    # 4) summaries apenas os suportados
    try:
        valid_sum = set(MFE.valid_summary())
    except Exception:
        valid_sum = {"mean","sd","median","min","max"}  # fallback seguro

    wanted_sum = ["mean","sd","median","min","max","quantiles"]
    summaries = [s for s in wanted_sum if s in valid_sum]
    if not summaries:
        summaries = ["mean","sd"]

    if verbose:
        print(f"[PyMFE] groups={groups}")
        print(f"[PyMFE] summaries={summaries}")
        print(f"[PyMFE] {len(feats)} features selecionadas (ex.: {sorted(feats)[:10]}...)")

    # 5) extrair com PyMFE (AGORA features != None)
    mfe = MFE(
        groups=groups,
        features=feats,
        summary=summaries,
        random_state=random_state,
        score="accuracy",
        num_cv_folds=3,
        lm_sample_frac=0.5,
    )
    mfe.fit(X_std, y)
    names, values = mfe.extract()

    # 6) montar DF + metadados
    meta = pd.Series(dict(zip(names, values)), dtype="float64")
    df = (
        meta.rename_axis("feature")
            .to_frame("value")
            .reset_index()
    )
    df["dataset"] = dataset_name
    df["split"] = split
    df["repr"] = repr_name
    df["n"] = int(n)
    df["d"] = int(d)

    # 7) salvar
    ensure_dir(out_dir)
    out_path = Path(out_dir) / f"{dataset_name}_{split}_{repr_name}.parquet"
    df.to_parquet(out_path, index=False)
    if verbose:
        print(f"✔ {repr_name}: {len(df)} meta-features salvas em {out_path.resolve()}")
    return df


In [20]:
# Usa as variáveis criadas nas suas células anteriores
X_bert = to_numpy(bert_features)
y_bert = to_numpy(bert_labels).astype(int).ravel()

X_roberta = to_numpy(roberta_features)
y_roberta = to_numpy(roberta_labels).astype(int).ravel()

print("BERT:", X_bert.shape, y_bert.shape, " | RoBERTa:", X_roberta.shape, y_roberta.shape)


BERT: (2490, 768) (2490,)  | RoBERTa: (2490, 768) (2490,)


In [21]:
df_meta_bert = extract_metafeatures_with_pymfe(
    X_bert, y_bert,
    dataset_name="RTE",
    split="train",
    repr_name="bert-base_pooler",     # ajuste o nome como preferir
    out_dir="meta",
    fast_mode=True,
    include_landmarkers=False,        # mude para True se quiser o “termômetro” 1-NN/LDA/NB/stump
)

df_meta_roberta = extract_metafeatures_with_pymfe(
    X_roberta, y_roberta,
    dataset_name="RTE",
    split="train",
    repr_name="roberta-base_classifier-dense",
    out_dir="meta",
    fast_mode=True,
    include_landmarkers=False,
)

# Visualização rápida
display(df_meta_bert.head(10))
display(df_meta_roberta.head(10))


[PyMFE] groups=['general', 'statistical', 'complexity']
[PyMFE] summaries=['mean', 'sd', 'median', 'min', 'max', 'quantiles']
[PyMFE] 23 features selecionadas (ex.: ['attr_to_inst', 'cor', 'cov', 'f1', 'f2', 'f3', 'f4', 'kurtosis', 'l3', 'mean']...)


/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1568: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1281: UserWarning:  * Something went wrong while precomputing 'precompute_can_cors'. Will ignore this method. Error message:
TypeError("OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'").
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'mean'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
 Will set it as 'np.nan' for all summary functions.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'mean' with summary 'quantiles'. Will set it as 'np.nan'.
  warni

✔ bert-base_pooler: 158 meta-features salvas em /home/dods/matrix/IC/Notebooks/meta/RTE_train_bert-base_pooler.parquet
[PyMFE] groups=['general', 'statistical', 'complexity']
[PyMFE] summaries=['mean', 'sd', 'median', 'min', 'max', 'quantiles']
[PyMFE] 23 features selecionadas (ex.: ['attr_to_inst', 'cor', 'cov', 'f1', 'f2', 'f3', 'f4', 'kurtosis', 'l3', 'mean']...)


/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:1281: UserWarning:  * Something went wrong while precomputing 'precompute_can_cors'. Will ignore this method. Error message:
TypeError("OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'").
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:606: RuntimeWarning: Can't extract feature 'mean'.
 Exception message: AttributeError('`np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.').
 Will set it as 'np.nan' for all summary functions.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/pymfe/_internal.py:731: RuntimeWarning: Can't summarize feature 'mean' with summary 'quantiles'. Will set it as 'np.nan'.
  warnings.warn(
/home/dods/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dods/minic

✔ roberta-base_classifier-dense: 158 meta-features salvas em /home/dods/matrix/IC/Notebooks/meta/RTE_train_roberta-base_classifier-dense.parquet


,feature,value,dataset,split,repr,n,d,n_classes,class_entropy,class_gini
0,attr_to_inst,0.308434,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
1,cor.max,0.999814,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
2,cor.mean,0.500893,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
3,cor.median,0.517741,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
4,cor.min,0.000003,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
5,cor.quantiles.0,0.000003,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
6,cor.quantiles.1,0.361915,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
7,cor.quantiles.2,0.517741,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
8,cor.quantiles.3,0.657965,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995
9,cor.quantiles.4,0.999814,RTE,train,bert-base_pooler,2490,768,2,0.693142,0.499995


,feature,value,dataset,split,repr,n,d,n_classes,class_entropy,class_gini
0,attr_to_inst,3.084337e-01,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
1,cor.max,6.230767e-01,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
2,cor.mean,1.074198e-01,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
3,cor.median,9.091321e-02,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
4,cor.min,7.167637e-07,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
5,cor.quantiles.0,7.167637e-07,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
6,cor.quantiles.1,4.312116e-02,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
7,cor.quantiles.2,9.091321e-02,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
8,cor.quantiles.3,1.550964e-01,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995
9,cor.quantiles.4,6.230767e-01,RTE,train,roberta-base_classifier-dense,2490,768,2,0.693142,0.499995


### Teste de predição do Modelo

In [1]:
import torch, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "textattack/roberta-base-RTE"   # RoBERTa já fine-tuned em RTE

# 1) Dataset
ds = load_dataset("glue", "rte")

# 2) Tokenizer (pares de sentenças)
tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
def preprocess(batch, max_len=128):
    return tok(batch["sentence1"], batch["sentence2"],
               truncation=True, padding="max_length", max_length=max_len)

enc = ds.map(preprocess, batched=True)

# 3) Formatação PyTorch
cols = ["input_ids", "attention_mask", "label"]
# Em RoBERTa normalmente não existe token_type_ids, mas deixamos o código robusto:
if "token_type_ids" in enc["validation"].features:
    cols.append("token_type_ids")
enc["validation"].set_format(type="torch", columns=cols)

val_loader = DataLoader(enc["validation"], batch_size=32, shuffle=False)

# 4) Modelo
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()

# 5) Avaliação
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        inputs = {k: v.to(device) for k, v in batch.items()
                  if k in ["input_ids", "attention_mask", "token_type_ids"]}
        logits = model(**inputs).logits
        preds  = logits.argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(batch["label"].numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

# 6) Métricas
acc = accuracy_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred, labels=[0, 1])
id2label = model.config.id2label
target_names = [id2label.get(i, str(i)) for i in [0, 1]]

print(f"RoBERTa RTE — Accuracy (validation): {acc:.4f}")
print("Confusion matrix [rows=true, cols=pred]:\n", cm)
print(classification_report(y_true, y_pred, target_names=target_names))
print(f"Cobertura: {len(y_pred)} de {len(enc['validation'])} exemplos")


/home/dods/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 3000/3000 [00:00<00:00, 16829.88 examples/s]
Some weights of the model checkpoint at textattack/roberta-base-RTE were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassi

RoBERTa RTE — Accuracy (validation): 0.7942
Confusion matrix [rows=true, cols=pred]:
 [[126  20]
 [ 37  94]]
              precision    recall  f1-score   support

     LABEL_0       0.77      0.86      0.82       146
     LABEL_1       0.82      0.72      0.77       131

    accuracy                           0.79       277
   macro avg       0.80      0.79      0.79       277
weighted avg       0.80      0.79      0.79       277

Cobertura: 277 de 277 exemplos
